# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units Sold: {total_units:,}")
print(f"Meaning: Over the 400 generated orders, the total revenue generated is ${total_revenue:,.2f} with a total of {total_units:,} units sold across all categories.")

Total Revenue: $8,520.00
Total Units Sold: 783
Meaning: Over the 400 generated orders, the total revenue generated is $8,520.00 with a total of 783 units sold across all categories.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')['revenue'].sum().reset_index()
by_category = by_category.sort_values(by='revenue', ascending=False)
by_category['percentage'] = (by_category['revenue'] / total_revenue) * 100

display(by_category)

print("\nMeaning:")
for _, row in by_category.iterrows():
    print(f"Category '{row['category']}' generated ${row['revenue']:.2f} in revenue, which accounts for {row['percentage']:.1f}% of total sales.")

,category,revenue,percentage
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986



Meaning:
Category 'Food' generated $4293.00 in revenue, which accounts for 50.4% of total sales.
Category 'Merch' generated $1771.50 in revenue, which accounts for 20.8% of total sales.
Category 'Drink' generated $1554.00 in revenue, which accounts for 18.2% of total sales.
Category 'RainGear' generated $901.50 in revenue, which accounts for 10.6% of total sales.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_stats = df.groupby('vendor_id').agg(
    avg_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).sort_values(by='avg_revenue', ascending=False)

display(vendor_stats)

# Identify the highest average
highest_vendor = vendor_stats.index[0]
highest_avg = vendor_stats.iloc[0]['avg_revenue']
highest_count = vendor_stats.iloc[0]['order_count']

print(f"\nMeaning: Vendor {highest_vendor} has the highest average order revenue of ${highest_avg:.2f} across {int(highest_count)} orders.")

,avg_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105



Meaning: Vendor V-01 has the highest average order revenue of $22.60 across 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_rev = df[df['category'] == 'Merch']['revenue'].sum()
merch_share = (merch_rev / total_revenue) * 100
print(f"Merch Share of Revenue: {merch_share:.1f}%")
print(f"Meaning: Merch category generates ${merch_rev:,.2f} which constitutes {merch_share:.1f}% of the overall total revenue.")

Merch Share of Revenue: 20.8%
Meaning: Merch category generates $1,771.50 which constitutes 20.8% of the overall total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# Perform left join and validate 'many_to_one'
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Find unmatched
unmatched_vendors = df[~df['vendor_id'].isin(vendor_names['vendor_id'])]['vendor_id'].unique()
print(f"Unmatched vendor ID(s) found in orders: {unmatched_vendors}")

# Verify lengths and revenue sums did not change
print(f"Original df row count: {len(df)} | Joined df row count: {len(joined)}")
print(f"Original revenue sum: ${df['revenue'].sum():.2f} | Joined revenue sum: ${joined['revenue'].sum():.2f}")

# Since V-18 is unmatched, we keep it in the dataset to preserve overall revenue but its name remains NaN.

Unmatched vendor ID(s) found in orders: ['V-18']
Original df row count: 400 | Joined df row count: 400
Original revenue sum: $8520.00 | Joined revenue sum: $8520.00


**The unmatched vendor, and what I did about it:** The unmatched vendor is `V-18`. I used a left join to keep all 400 orders in the DataFrame, meaning the unmatched orders remain with `NaN` as their vendor name. This ensures that the overall revenue metrics and row count do not change, keeping the dataset integral.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot_rep = df.pivot_table(
    index='vendor_id',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
display(pivot_rep)
print("\nMeaning: V-18 had the highest total revenue at $2,349, while Food led category revenue at $4,293. The grand total across all vendors and categories was $8,520.")

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0



Meaning: V-18 had the highest total revenue at $2,349, while Food led category revenue at $4,293. The grand total across all vendors and categories was $8,520.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
assert np.isclose(joined['revenue'].sum(), df['revenue'].sum()), 'the vendor merge changed total revenue'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** For the next game, I would prioritize maintaining enough Food inventory because it generated \$4,293.00, or 50.4% of total revenue. Merch was the second-largest category at \$1,771.50, or 20.8%, so vendors should also give it attention when planning stock. RainGear contributed \$901.50, or 10.6%, but I would check the weather forecast, leftover inventory, and profit margins before recommending a reduction. These results describe this generated dataset and provide a starting point for planning rather than a guarantee of future demand.

**b)** Q5 is the least trustworthy for identifying vendors by name because the lookup is missing `V-18`, which accounts for 108 orders and \$2,349.00 in revenue. The left join preserves those orders and their revenue, so the totals and the Q6 report grouped by vendor ID remain correct. However, grouping by vendor name with pandas' default settings would exclude the rows whose names are missing. I kept those rows with missing names rather than inventing a vendor name; the lookup should be completed before producing a report grouped by vendor name.